# Adding labels to ome-zarrs
(advanced:labels:adding_labels)=

Unlinke demonstrated in the [basic labels example](basic:labels), in many scenarios, it is nt desirable to write a complete structure consisting of an ome-zarr image and some labels at once.
Instead, one might want to *first* write the ome-zarr image (i.e., after image conversion), and *then* add the labels later (i.e., after segmentation).

The ome-zarr-py allows to do this in a lean and top-level way.

In [1]:
import numpy as np
from skimage.data import binary_blobs
from ome_zarr import NgffImage, NgffMultiscales

First, let's create some sample ome-zarr image data and write it to disk:

In [2]:
rng = np.random.default_rng(0)
data = rng.poisson(10, size=(64, 64, 64)).astype(np.uint8)

ngff_image = NgffImage(data, axes="zyx", name="image")
ngff_multiscales = NgffMultiscales(image=ngff_image)

# write the image data to disk
ngff_multiscales.to_ome_zarr("image_with_labels.zarr")

[]

## Adding labels data

Now, we perform a pseudo-segmentation and create some labels data.
The created data is then also converted into an instance of {py:class}`ome_zarr.NgffMultiscales`.

In [3]:
labels = NgffImage(
    data=binary_blobs(length=64, volume_fraction=0.1, n_dim=3).astype('int8'),
    axes="zyx",
)
labels_multiscales = NgffMultiscales(image=labels)

If we assume that the image data has been written to disk previously, we need to open the existing ome-zarr file to append our labels data to the `labels` attribute of the parent image.
Since no label images have yet been added to the group, we find the `labels` attribute ampty:

In [4]:
parent_image = NgffMultiscales.from_ome_zarr("image_with_labels.zarr")
print(parent_image.labels)

None


 This attribute is going to be a Python dictionary, so new images can be added to it as follows:

In [5]:
parent_image.labels = {
    "label_image_1": labels_multiscales
  }

We can now store the additional labels data to the already existing group under `"image_with_labels.zarr/labels/label_image_1"`. The name of the label image group corresponds to the key of the dictionary entry above.

```{note}
When writing the labels data, we want to **append** the labels data, not write the entire image data again. For this to happen, we need to set the `overwrite` argument of the `to_ome_zarr` method to `False`
```



In [6]:
parent_image.to_ome_zarr("image_with_labels.zarr", overwrite=False)

[]

## Appending more labels data

In the above example, we added a single label image to the parent image.
However, it is possible to distinctively append labels images rather than adding them alltogether. If we open the ome-zarr file again, we can find the already added label image under the `labels` attribute of the parent image:

In [8]:
parent_image = NgffMultiscales.from_ome_zarr("image_with_labels.zarr")
parent_image.labels

{'label_image_1': NgffMultiscales(images=[NgffImage(data=dask.array<from-zarr, shape=(64, 64, 64), dtype=int8, chunksize=(64, 64, 64), chunktype=numpy.ndarray>, axes=['z', 'y', 'x'], scale={'z': 1.0, 'y': 1.0, 'x': 1.0}, axes_units=None, name='image'), NgffImage(data=dask.array<from-zarr, shape=(64, 32, 32), dtype=int8, chunksize=(64, 32, 32), chunktype=numpy.ndarray>, axes=['z', 'y', 'x'], scale={'z': 1.0, 'y': 2.0, 'x': 2.0}, axes_units=None, name='image'), NgffImage(data=dask.array<from-zarr, shape=(64, 16, 16), dtype=int8, chunksize=(64, 16, 16), chunktype=numpy.ndarray>, axes=['z', 'y', 'x'], scale={'z': 1.0, 'y': 4.0, 'x': 4.0}, axes_units=None, name='image'), NgffImage(data=dask.array<from-zarr, shape=(64, 8, 8), dtype=int8, chunksize=(64, 8, 8), chunktype=numpy.ndarray>, axes=['z', 'y', 'x'], scale={'z': 1.0, 'y': 8.0, 'x': 8.0}, axes_units=None, name='image'), NgffImage(data=dask.array<from-zarr, shape=(64, 4, 4), dtype=int8, chunksize=(64, 4, 4), chunktype=numpy.ndarray>, axe

We can now append more label image to the `labels` attribute of the parent image...

In [9]:
labels2 = NgffImage(
    data=binary_blobs(length=64, volume_fraction=0.1, n_dim=3).astype('int8'),
    axes="zyx",
)
labels_multiscales2 = NgffMultiscales(image=labels2)

...and write the whole lot to disk.
Again, when `overwrite=False`, the existing image data (and the already written label image `label_image_1`) will not be overwritten, but the new label image `label_image_2` will be added to the `labels` group as a new subgroup. 

In [ ]:
parent_image.labels["label_image_2"] = labels_multiscales2
parent_image.to_ome_zarr("image_with_labels.zarr", overwrite=False)